# Counting vehicles by class for signal timing (UVH-26, YOLOv8)

AIML ZG525 Computer Vision, EC-1 Assignment 1 · Track B (object detection)

This notebook runs top to bottom on a free Colab T4 (*Runtime → Change runtime type → T4 GPU*).

- **`RUN_TRAINING`** (next cell): `True` retrains both models; `False` skips training and evaluates the weights saved on Drive.
- The dataset is downloaded by the notebook from the Hugging Face Hub at a pinned revision; no data files are embedded.
- Sections follow the report's 8 steps. Every number quoted in the report appears in a cell output below.
- Optional Colab secrets: `HF_TOKEN` (faster Hub downloads), `GITHUB_TOKEN` (only if the repo is private).

In [ ]:
# ---- Run settings: the only cell a grader should need to edit ----
RUN_TRAINING = True   # False: skip training, evaluate the weights saved under DRIVE_ROOT/runs
REPO_URL = "https://github.com/YashRenwa/uvh26-pcu-counting.git"
COMMIT = "main"       # pinned to a commit hash for the final submission run
DRIVE_ROOT = "/content/drive/MyDrive/uvh26"

In [ ]:
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    repo_dir = Path("/content/uvh26-pcu-counting")
    if not repo_dir.exists():
        try:
            token = userdata.get("GITHUB_TOKEN")  # only needed for a private repo
        except Exception:
            token = None
        url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
        subprocess.run(f"git clone -q {url} {repo_dir}", shell=True, check=True)
    os.chdir(repo_dir)
    # Always run exactly what is on GitHub: a branch name follows origin, a hash is used as is.
    # -f discards edits made inside this Colab clone (e.g. regenerated results/); Drive is untouched.
    sh("git fetch -q origin")
    is_branch = subprocess.run(f"git rev-parse -q --verify origin/{COMMIT}", shell=True, capture_output=True).returncode == 0
    sh(f"git checkout -q -f --detach {'origin/' + COMMIT if is_branch else COMMIT}")
    # Keep Colab's preinstalled torch/opencv/numpy (CUDA-matched, no restart); pin the rest.
    sh("grep -v colab-skip requirements.txt > /tmp/req-colab.txt && pip install -q -r /tmp/req-colab.txt")
    os.environ.update({
        "UVH_RAW_ROOT": "/content/raw",              # original PNGs on fast local disk
        "UVH_DATA_ROOT": f"{DRIVE_ROOT}/prepared",   # JPEGs + labels survive a disconnect
        "UVH_RUNS_ROOT": f"{DRIVE_ROOT}/runs",
        "UVH_WEIGHTS_DIR": f"{DRIVE_ROOT}/weights",
        "UVH_DATASETS_ROOT": "/content/datasets",   # training reads from local disk, not Drive
        "UVH_RESULTS_DIR": f"{DRIVE_ROOT}/results",  # results/*.json and figures survive the session
    })
else:
    repo_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(repo_dir)

sys.path.insert(0, str(repo_dir))
commit = subprocess.run("git rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip()
print("repo:", repo_dir, "| commit:", commit or "(no commits yet)")

In [ ]:
import torch
from src.common import library_versions, load_config, set_seed

SMOKE = os.environ.get("UVH_SMOKE") == "1"  # local CPU smoke test on 20 images; never set on Colab
cfg = load_config(smoke=SMOKE)
set_seed(cfg["seed"])
print("seed:", cfg["seed"])
for name, version in library_versions().items():
    print(f"  {name:16s} {version}")
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("paths:", {k: str(v) for k, v in cfg["paths"].items()})

## 1. Problem and task type

*Written with the report.* Signal timing needs per-class vehicle **counts** at a junction, converted to PCU. Counting needs instances, so this is object detection: classification cannot count, and segmentation adds labelling and compute cost without changing the count.

## 2. Data and split  *(Phase 1)*

**Source.** UVH-26 (IISc AIM with the Bengaluru Traffic Police), **CC BY 4.0**, downloaded from the Hugging Face Hub at the revision pinned in `configs/base.yaml`. The full set is 26,646 PNGs (90 GB), so only the four annotation JSONs and a seeded pool of 4,000 images are fetched (`splits/pool.csv`).

**Pool.** STAPLE labels exist for only 81% of images (MV covers all), so the pool is drawn from images that carry both label sets. The MV and STAPLE runs then train on identical images.

**Pseudo-cameras.** UVH-26 has no camera ids, and a fixed CCTV camera produces many near-identical frames. Two images are linked when the correlation **g** of their 64×36 grayscale *gradient-magnitude* thumbnails is ≥ 0.48; connected components are pseudo-cameras. A fixed camera keeps its static edges at the same pixels, so g survives lighting, glare and colour-vs-IR changes. CLIP ViT-B/32 cosine and intensity correlation did not: CLIP alone chained different cameras into giant groups, and on 88 hand-labelled hard pairs g separated cameras best (AUC 0.97 vs 0.80 for CLIP).

**Split.** Whole pseudo-cameras are split 60/20/20 with the best class balance among 50 seeds, then a **purge** moves any val/test image with a train image at g ≥ 0.36 (and its pseudo-camera) into train, repeated until none is left. A random audit of 40 test images still finds a same-camera train image for about 7 of them, reported as residual leakage.

**Frozen split.** The split was built once and committed (`splits/`, checksum in `splits/test.sha256`). CLIP outputs differ slightly between CPU and GPU, so this notebook recomputes the grouping to show the evidence, but trains and evaluates on the committed split after verifying the test-set checksum.

In [ ]:
import pandas as pd
from IPython.display import Image, display
from src import cluster, download, prepare, split

FIGS = cfg["paths"]["results_dir"] / "figures"
pool = download.select_pool(cfg)   # reads splits/pool.csv from the repo
print(f"pool: {len(pool)} images with both MV and STAPLE labels")
print(pool.groupby(["official_split", "subfolder"]).size().rename("images").to_string())
download.download_pool(cfg, pool)  # PNGs to local disk; skips images already prepared on Drive

Resize to 1280 px on the long side (JPEG quality 95) and write YOLO labels twice, once from MV and once from STAPLE. YOLO box coordinates are normalised, so they need no rescaling.

In [ ]:
prep = prepare.run(cfg, pool)
print({k: v for k, v in prep.items() if k not in ("class_counts", "subfolders")})
display(pd.DataFrame(prep["class_counts"]).rename(columns=str.upper))
display(Image(filename=str(prepare.draw_labels(cfg, pool["stem"].iloc[0], FIGS / "labels_mv_vs_st.jpg")), width=900))

### Pseudo-cameras and the official split's leakage

In [ ]:
clu = cluster.run(cfg)
pc, lk, cal = clu["pseudo_cameras"], clu["leakage"], clu["calibration"]
print(f"linking rule: {clu['rule']} | {pc['groups']} pseudo-cameras | {pc['singletons']} singletons | largest {pc['largest_group']} images")
print(f"calibration (equal cameras): {cal['groups']} groups, {cal['singletons_pct_of_pool']}% singletons")
print(f"LEAKAGE: {lk['val_images_sharing_camera_with_train']} of {lk['val_images']} official-val images "
      f"({lk['leakage_val_share']:.1%}) share a pseudo-camera with an official-train image")
if clu["frozen_groups_used"]:
    print("frozen groups used; recomputed grouping identical:", clu["recomputed_groups_identical"])
display(pd.DataFrame(clu["rule_sweep"]).set_index("setting"))

In [ ]:
rule = cluster.Rule.from_config(cfg).name
for name, width in (("nn_gradient_g.png", 640), ("group_sizes.png", 640), (f"{rule}_above.jpg", 900),
                    (f"{rule}_below.jpg", 900), (f"{rule}_weakest_in_group.jpg", 900), ("leak_examples.jpg", 900)):
    if (FIGS / name).exists():  # a grid is skipped when no pair qualifies
        display(Image(filename=str(FIGS / name), width=width))

### Group split and class balance

In [ ]:
if (cfg["paths"]["splits_dir"] / "test.sha256").exists():
    print("split is frozen; test.txt sha256 verified:", split.check_test_checksum(cfg))
    assignment = split.load(cfg)
else:
    assignment = split.make(cfg)
spl = split.summarise(cfg, assignment)
floor = cfg["split"]["min_test_instances"]
for k in ("images", "image_share", "pseudo_cameras", "official_split_composition",
          "classes_missing", f"classes_under_{floor}_test_instances"):
    print(f"{k}: {spl[k]}")
display(pd.read_csv(cfg["paths"]["results_dir"] / "split_class_counts.csv", header=[0, 1], index_col=0))

## 3. Preprocessing and augmentation  *(Phases 1-2)*

## 4. Baseline: YOLOv8s fine-tuned on MV labels  *(Phase 2)*

### Context: COCO-pretrained YOLOv8s as-is
No fine-tuning. Predictions and labels are mapped to shared coarse classes (cars → car; Bus, Mini-bus → bus; Truck, LCV → truck; Two-wheeler → motorcycle; Bicycle → bicycle). Three-wheeler, Tempo-traveller, Van and Others have no COCO class and count as misses. This shows why fine-tuning is needed. Val only.

In [ ]:
from src import evaluate, train

coco = evaluate.coco_context(cfg)
for ls in ("mv", "st"):
    m = coco[ls]
    print(f"{ls.upper()} labels: mAP50 {m['mAP50']:.3f}, mAP50-95 {m['mAP50_95']:.3f} over all coarse classes | "
          f"mappable only: mAP50 {m['mAP50_mappable_only']:.3f}, mAP50-95 {m['mAP50_95_mappable_only']:.3f}")
display(pd.DataFrame(coco["mv"]["per_class"]).T)

### Freeze pilot (val only)
YOLOv8s at 960 px, batch 8, seed 42, MV labels, `pilot_epochs` epochs per arm: backbone frozen (`freeze=10`) vs fully trainable. The choice is made on val mAP@0.50:0.95 and minutes per epoch. Runs are saved to Drive; re-running this cell resumes an interrupted arm and skips a finished one.

In [ ]:
if RUN_TRAINING:
    for arm in ("frozen", "full"):
        train.ensure_run(cfg, "mv", pilot=arm)
pilot = train.pilot_summary(cfg)
display(pd.DataFrame({arm: {
    "val mAP50-95": s["val_last"]["mAP50_95"], "val mAP50": s["val_last"]["mAP50"],
    "minutes / epoch": s["minutes_per_epoch"], "first epoch (min)": s["minutes_first_epoch"],
    "total (min)": s["minutes_total"], f"epochs within {pilot['budget_minutes']} min": s["epochs_within_budget"],
} for arm, s in pilot["arms"].items()}))

### Epoch budget
One full run must fit the 90-minute budget on a free T4, with the same epoch count for baseline and ablation. The pilot (5 epochs) ran entirely without mosaic, because Ultralytics switches mosaic off for the last `close_mosaic` = 10 epochs. A full run uses mosaic before that, and mosaic costs 1.35× per sample in the data pipeline (measured), so those epochs are budgeted as 1.35× slower (worst case: the run is data-bound). A 10% margin covers setup and the final validation. The chosen values are written to `configs/base.yaml` (`train.epochs`, `train.freeze`).

In [ ]:
t = cfg["train"]
print(f"config: train.freeze = {t['freeze']}, train.epochs = {t['epochs']} "
      f"(close_mosaic {t['close_mosaic']}, mosaic cost x{t['mosaic_slowdown']}, budget {t['budget_minutes']} min)")
for arm, s in pilot["arms"].items():
    print(f"  {arm:6s}: freeze = {train.PILOTS[arm]}, {s['minutes_per_epoch']:.2f} min/epoch without mosaic -> "
          f"{s['epochs_within_budget']} epochs fit the budget (worst case); "
          f"{t['epochs']} epochs take at most {s['worst_case_minutes_at_config_epochs']} min")

### Baseline run
YOLOv8s, COCO weights, all layers trainable, 960 px, batch 8, seed 42, `train.epochs` epochs with early stopping (patience 10), MV labels. The ablation (Section 5) runs the same call with `"st"`; nothing else differs. The run is saved to Drive every epoch (`last.pt`) and every 5 epochs; re-running this cell after a disconnect resumes it.

In [ ]:
if RUN_TRAINING:
    baseline_dir = train.ensure_run(cfg, "mv")
else:
    baseline_dir = cfg["paths"]["runs_root"] / "mv"
baseline = train.run_summary(baseline_dir)
print({k: baseline[k] for k in ("epochs_run", "minutes_total", "minutes_per_epoch")})
print("val during training, best epoch:", baseline["val_best"])
display(Image(filename=str(baseline_dir / "results.png"), width=900))

### Generalisation gap
The baseline's best weights (`best.pt`, chosen on val mAP@0.50:0.95) scored on the train split and on val, MV labels, with identical settings (conf 0.001, NMS IoU 0.7, 960 px). Test stays sealed.

In [ ]:
best = baseline_dir / "weights" / "best.pt"
gap = {s: evaluate.detector_metrics(cfg, best, "mv", s, name=f"mv__{s}__mv") for s in ("train", "val")}
overall = pd.DataFrame({s: {k: m[k] for k in ("mAP50", "mAP50_95", "precision", "recall")} for s, m in gap.items()})
display(overall.assign(gap=overall["train"] - overall["val"]).round(3))
per_class = pd.DataFrame({s: {c: v["AP50_95"] for c, v in m["per_class"].items()} for s, m in gap.items()})
per_class["val instances"] = pd.Series({c: v["instances"] for c, v in gap["val"]["per_class"].items()})
display(per_class.assign(gap=per_class["train"] - per_class["val"]).round(3).rename_axis("AP50-95"))

### Fine-tuning gain, like for like
The baseline's 14-class predictions mapped to the same coarse classes as the COCO-as-is run (cars → car, Bus/Mini-bus → bus, Truck/LCV → truck, …; NMS re-run per coarse class), scored with the same matching on val. This is the fair comparison with the COCO context number above.

In [ ]:
fine = evaluate.finetuned_coarse(cfg, best, "mv")
rows = {"mAP50-95, all 9 coarse classes": "mAP50_95", "mAP50-95, 5 COCO-mappable": "mAP50_95_mappable_only",
        "mAP50, all 9 coarse classes": "mAP50", "mAP50, 5 COCO-mappable": "mAP50_mappable_only"}
display(pd.DataFrame({"COCO as-is": {r: coco["mv"][k] for r, k in rows.items()},
                      "fine-tuned on MV": {r: fine["mv"][k] for r, k in rows.items()}}).round(3))
display(pd.DataFrame({"COCO as-is": {c: v["AP50_95"] for c, v in coco["mv"]["per_class"].items()},
                      "fine-tuned on MV": {c: v["AP50_95"] for c, v in fine["mv"]["per_class"].items()}}).round(3).rename_axis("AP50-95"))

## 5. Ablation: MV vs STAPLE training labels  *(Phase 3)*

The crowd labels were merged two ways: majority voting (MV) and STAPLE (ST, an expectation-maximisation consensus). The ablation trains a second model **identical to the baseline except for the training labels**: same config file, seed 42, 34 epochs, all layers trainable, 960 px, same train and val images. Only `train.ensure_run(cfg, "st")` differs from the baseline call.

In [ ]:
if RUN_TRAINING:
    ablation_dir = train.ensure_run(cfg, "st")
else:
    ablation_dir = cfg["paths"]["runs_root"] / "st"
ablation = train.run_summary(ablation_dir)
print({k: ablation[k] for k in ("epochs_run", "minutes_total", "minutes_per_epoch")})
print("val during training (ST labels), best epoch:", ablation["val_best"])
display(Image(filename=str(ablation_dir / "results.png"), width=900))

### Proof of control
Every training setting of the two runs, compared from the settings stored in each run's checkpoint (after Ultralytics resolves `auto` values). Only the data yaml, i.e. the label set, may differ, plus run bookkeeping (name, output folder, resume paths). The raw `args.yaml` diff is saved too; a resumed run rewrites `args.yaml` with resolved values, so the checkpoint settings are the reliable record.

In [ ]:
ctrl = train.args_diff(cfg, "mv", "st")
display(pd.DataFrame(ctrl["effective_differing_keys"], index=["MV run", "STAPLE run"]).T)
print("label-set keys:", ctrl["label_set_keys"], "| bookkeeping:", ctrl["bookkeeping_keys"], "| unexpected:", ctrl["unexpected_keys"])
print("same train / val images:", ctrl["same_images"], "| resumed:", ctrl["resumed"])
print("ONLY DIFFERENCE IS THE LABEL SET:", ctrl["only_difference_is_label_set"])

### 2 × 2: trained-on × scored-against (val)
Scoring each model only against its own label style is biased: a box one consensus keeps and the other drops counts as a miss or a false positive depending on the labels used. Each model is therefore scored against both MV and STAPLE val labels. Compare models **within a column** (same labels).

In [ ]:
runs = {"mv": baseline_dir, "st": ablation_dir}
cells = {(tr, sc): evaluate.detector_metrics(cfg, runs[tr] / "weights" / "best.pt", sc, "val", name=f"{tr}__val__{sc}")
         for tr in ("mv", "st") for sc in ("mv", "st")}
for metric in ("mAP50_95", "mAP50", "precision", "recall"):
    print(metric)
    display(pd.DataFrame({f"scored vs {sc.upper()} labels": {f"trained on {tr.upper()}": cells[(tr, sc)][metric]
                                                              for tr in ("mv", "st")} for sc in ("mv", "st")}))
per = pd.DataFrame({f"{tr.upper()}-trained vs {sc.upper()}": {c: v["AP50_95"] for c, v in cells[(tr, sc)]["per_class"].items()}
                    for tr, sc in cells})
per["val instances (MV)"] = pd.Series({c: v["instances"] for c, v in cells[("mv", "mv")]["per_class"].items()})
display(per.round(3).rename_axis("AP50-95"))

### Is the difference larger than evaluation noise?
Paired bootstrap over val images: the same resampled image sets are scored for both models, giving a 95% interval for (STAPLE-trained − MV-trained) in each column. This measures evaluation-set sampling noise only; seed-to-seed training noise would need a second pair of runs.

In [ ]:
boot = {sc: evaluate.paired_bootstrap(cfg, f"mv__val__{sc}", f"st__val__{sc}", n_boot=cfg["evaluate"]["n_boot"])
        for sc in ("mv", "st")}
display(pd.DataFrame({f"scored vs {sc.upper()} labels": {
    "MV-trained mAP50-95": b["a_mAP50_95"], "ST-trained mAP50-95": b["b_mAP50_95"],
    "ST - MV, mAP50-95": b["diff_mAP50_95"], "95% CI": b["diff_mAP50_95_ci95"],
    "ST - MV, mAP50": b["diff_mAP50"], "95% CI ": b["diff_mAP50_ci95"],
    "resamples where ST better (mAP50-95)": b["share_b_better_mAP50_95"],
} for sc, b in boot.items()}))

### Which classes move?
Per-class AP@0.50:0.95 difference (STAPLE-trained − MV-trained) with 95% intervals from the same paired bootstrap, in each column. Classes with fewer than about 30 val instances (Mini-bus, Tempo-traveller, Others) are too small to read.

In [ ]:
per_cls = {}
for sc in ("st", "mv"):
    for c, v in boot[sc]["per_class_AP50_95"].items():
        per_cls.setdefault(c, {})[f"ST - MV vs {sc.upper()}"] = v["diff"]
        per_cls[c][f"95% CI vs {sc.upper()}"] = v["ci95"]
per_cls = pd.DataFrame(per_cls).T
per_cls["val instances (ST)"] = pd.Series({c: v["instances"] for c, v in cells[("st", "st")]["per_class"].items()})
display(per_cls.rename_axis("AP50-95"))

### Robustness: last-epoch weights
Each run keeps the checkpoint with the best val mAP against its *own* labels: the MV run picked epoch 34 (its last), the STAPLE run epoch 25. Choosing on the scoring labels slightly favours the STAPLE model in the STAPLE column. Scoring both models' epoch-34 weights (`last.pt`, no selection) removes that asymmetry.

In [ ]:
last = {(tr, sc): evaluate.detector_metrics(cfg, runs[tr] / "weights" / "last.pt", sc, "val", name=f"{tr}_last__val__{sc}")
        for tr in ("mv", "st") for sc in ("mv", "st")}
boot_last = {sc: evaluate.paired_bootstrap(cfg, f"mv_last__val__{sc}", f"st_last__val__{sc}", n_boot=cfg["evaluate"]["n_boot"])
             for sc in ("mv", "st")}
display(pd.DataFrame({f"scored vs {sc.upper()} labels": {
    "MV-trained, epoch 34": b["a_mAP50_95"], "ST-trained, epoch 34": b["b_mAP50_95"],
    "ST - MV, mAP50-95": b["diff_mAP50_95"], "95% CI": b["diff_mAP50_95_ci95"],
} for sc, b in boot_last.items()}))

## 6. Evaluation  *(Phase 4 threshold on val; Phase 5 sealed test, once)*

### From detections to Passenger Car Units  *(Phase 4, val only)*
Green time is allocated by demand in PCU, so each detected vehicle is weighted by its IRC:106-1990 equivalency factor. IRC gives a factor at 5% and at 10%-and-above share of the traffic stream; the shares come from the training split's STAPLE labels and factors are interpolated linearly in between. All four car subtypes and vans weigh 1.0 PCU, so confusing a Hatchback with an SUV costs nothing here; confusing a Truck (3.27) with an LCV (1.57) does.

In [ ]:
from src import counts

fac = counts.pcu_factors(cfg)
display(pd.DataFrame({"IRC category": fac["class_category"], "PCU factor": fac["class_factor"]})
        .assign(**{"category share of train traffic (%)": lambda d: d["IRC category"].map(fac["category_share_pct"])}))

### Operating threshold
Counting uses one box per vehicle: each box keeps its top class and NMS is class-agnostic (the mAP evaluation keeps several labels per box, which would count one vehicle more than once). For each model, the confidence threshold is swept from 0.05 to 0.70 on val and the value with the lowest mean absolute PCU error per image is chosen, against STAPLE labels (the headline label set). The chosen values are then frozen in `configs/base.yaml` (`evaluate.conf`) before the test set is opened.

In [ ]:
cnt = counts.run(cfg)
ref = cnt["reference_labels"].upper()
print(f"val: {cnt['images']} images, mean true demand {cnt['true_pcu_per_image_mean']} PCU per image ({ref} labels)")
display(Image(filename=str(FIGS / "pcu_threshold_sweep_val.png"), width=900))
display(pd.DataFrame({f"trained on {m.upper()}": {
    "chosen threshold": r["chosen_threshold"], "frozen in config": r["frozen_conf"],
    **{f"{k} vs {ls.upper()}": v for ls in ("st", "mv") for k, v in r["at_threshold"][ls].items()},
} for m, r in cnt["models"].items()}))
for m, r in cnt["models"].items():
    print(f"{m.upper()}-trained model at threshold {r['chosen_threshold']}: per-class counts vs {ref} labels")
    display(pd.DataFrame(r["per_class_vs_reference"]).T)

## 7. Error analysis  *(Phase 6)*

## 8. Conclusion